In [ ]:
import sys
sys.path.append("src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import quantum_algorithms as qa

In [ ]:
# =========================
# Full Master Thesis Experiment
# =========================

f_const = qa.deutsch_jozsa.f_constant_1
f_bal = qa.deutsch_jozsa.f_balanced_parity

error_positions = {
    "E1_before_H": qa.deutsch_jozsa.deutsch_jozsa_error1,
    "E2_after_first_H": qa.deutsch_jozsa.deutsch_jozsa_error2,
    "E3_after_oracle": qa.deutsch_jozsa.deutsch_jozsa_error3,
    "E4_after_final_H": qa.deutsch_jozsa.deutsch_jozsa_error4,
}

label_map = {
    "no_error": "No Error",
    "E1_before_H": r"$\mathcal{E}_1$",
    "E2_after_first_H": r"$\mathcal{E}_2$",
    "E3_after_oracle": r"$\mathcal{E}_3$",
    "E4_after_final_H": r"$\mathcal{E}_4$",
}

n_values = range(1, 13)                    # n = 1 to 12
theta_values = np.linspace(0, np.pi, 181)  # 0° to 180°, step 1°

axes = {
    "X": (1, 0, 0),
    "Y": (0, 1, 0),
    "Z": (0, 0, 1),
}

shots = 1024
rng = np.random.default_rng(12345)

results = []

In [ ]:
# =========================
# Run full experiment
# =========================

for n in n_values:
    print(f"Running n = {n}")

    for theta in theta_values:
        theta_deg = np.degrees(theta)

        for axis_name, axis in axes.items():
            for target_qubit in range(n):

                # Original DJA
                state_const_ref = qa.deutsch_jozsa.deutsch_jozsa(n, f_const)
                state_bal_ref = qa.deutsch_jozsa.deutsch_jozsa(n, f_bal)

                samples_const_ref = qa.deutsch_jozsa.sample_measurements_input(
                    state_const_ref, n, shots, rng
                )
                samples_bal_ref = qa.deutsch_jozsa.sample_measurements_input(
                    state_bal_ref, n, shots, rng
                )

                P0_constant_ref = samples_const_ref[0] / shots
                P0_balanced_ref = samples_bal_ref[0] / shots

                results.append({
                    "n": n,
                    "theta_rad": theta,
                    "theta_deg": theta_deg,
                    "axis": axis_name,
                    "target_qubit": target_qubit,
                    "error_position": "no_error",
                    "P0_constant": P0_constant_ref,
                    "P0_balanced": P0_balanced_ref,
                    "success_constant": P0_constant_ref,
                    "success_balanced": 1 - P0_balanced_ref,
                    "average_success": (P0_constant_ref + (1 - P0_balanced_ref)) / 2,
                    "shots": shots,
                })

                # Error cases E1 to E4
                for error_name, error_function in error_positions.items():

                    state_const = error_function(
                        n, f_const, theta, target_qubit, axis
                    )

                    state_bal = error_function(
                        n, f_bal, theta, target_qubit, axis
                    )

                    samples_const = qa.deutsch_jozsa.sample_measurements_input(
                        state_const, n, shots, rng
                    )
                    samples_bal = qa.deutsch_jozsa.sample_measurements_input(
                        state_bal, n, shots, rng
                    )

                    P0_constant = samples_const[0] / shots
                    P0_balanced = samples_bal[0] / shots

                    results.append({
                        "n": n,
                        "theta_rad": theta,
                        "theta_deg": theta_deg,
                        "axis": axis_name,
                        "target_qubit": target_qubit,
                        "error_position": error_name,
                        "P0_constant": P0_constant,
                        "P0_balanced": P0_balanced,
                        "success_constant": P0_constant,
                        "success_balanced": 1 - P0_balanced,
                        "average_success": (P0_constant + (1 - P0_balanced)) / 2,
                        "shots": shots,
                    })

df = pd.DataFrame(results)
df